In [1]:
YOUR_NAME = 'sara'

AWS_PROFILE = 'cities'

In [2]:
%load_ext autoreload

In [3]:
MAIN_PATH = "s3://wri-cities-sandbox/identifyingLandSubdivisions/data"
INPUT_PATH = f'{MAIN_PATH}/input'
CITY_INFO_PATH = f'{INPUT_PATH}/city_info'
EXTENTS_PATH = f'{CITY_INFO_PATH}/extents'
BUILDINGS_PATH = f'{INPUT_PATH}/buildings'
BLOCKS_PATH = f'{INPUT_PATH}/blocks'
ROADS_PATH = f'{INPUT_PATH}/roads'
INTERSECTIONS_PATH = f'{INPUT_PATH}/intersections'
GRIDS_PATH = f'{INPUT_PATH}/city_info/grids'
SEARCH_BUFFER_PATH = f'{INPUT_PATH}/city_info/search_buffers'
OUTPUT_PATH = f'{MAIN_PATH}/output'
OUTPUT_PATH_CSV = f'{OUTPUT_PATH}/csv'
OUTPUT_PATH_RASTER = f'{OUTPUT_PATH}/raster'
OUTPUT_PATH_PNG = f'{OUTPUT_PATH}/png'
OUTPUT_PATH_RAW = f'{OUTPUT_PATH}/raw_results'

In [4]:
# Check s3 connection using AWS_PROFILE=CitiesUserPermissionSet profile 
import boto3
boto3.setup_default_session(profile_name='cities')
session = boto3.Session(profile_name=AWS_PROFILE)
s3 = session.client('s3')

# export CitiesUserPermissionSet profile to use in the next cells
import os
os.environ['AWS_PROFILE'] = AWS_PROFILE

s3.list_buckets()

{'ResponseMetadata': {'RequestId': 'FJNAQ84Q7D9RDHPR',
  'HostId': 'PaWH7e0Obi0n6KY0AtMFBjXZ0gAGdDwMp5PIxt/WFMIhvzAwZXk7jB4c9nMRPQi3wVm2+cw6Zi8=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'PaWH7e0Obi0n6KY0AtMFBjXZ0gAGdDwMp5PIxt/WFMIhvzAwZXk7jB4c9nMRPQi3wVm2+cw6Zi8=',
   'x-amz-request-id': 'FJNAQ84Q7D9RDHPR',
   'date': 'Thu, 19 Mar 2026 21:07:32 GMT',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Buckets': [{'Name': 'aft-sandbox-540362055257',
   'CreationDate': datetime.datetime(2022, 9, 13, 15, 12, 20, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::aft-sandbox-540362055257'},
  {'Name': 'amplify-citiesindicatorsapi-dev-10508-deployment',
   'CreationDate': datetime.datetime(2023, 8, 30, 5, 5, 13, tzinfo=tzutc()),
   'BucketArn': 'arn:aws:s3:::amplify-citiesindicatorsapi-dev-10508-deployment'},
  {'Name': 'cities-heat',
   'CreationDate': datetime.datetime(2023, 6, 1, 13, 22, 1, tzinfo=tzutc

In [5]:
import dask_geopandas as dgpd
import dask.dataframe as dd
import pandas as pd
from dask import delayed, compute, visualize
import geopandas as gpd
from dask.diagnostics import ProgressBar
from shapely.geometry import MultiLineString, LineString, Point
from shapely.ops import polygonize, nearest_points
#from shapely.geometry import Polygon, LineString, Point, MultiPolygon, MultiLineString, GeometryCollection
from scipy.optimize import fminbound, minimize
#from unused_code.metrics_groupby import metrics
from dask import delayed
import dask_geopandas as dgpd
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely import wkb
from scipy.stats import entropy
import time
from dask import compute

from pre_processing import *
from auxiliary_functions import *
from standardize_metrics import *
from metrics_calculation import *

import shutil

import time

YOUR_NAME = 'sara'

In [ ]:
# CALCULATE BUILDING DISTANCES TO ROADS AND PRODUCE BLOCKS.

start_time = time.time()  # Start the timer

%autoreload
#from main import *

def preprocess_all_cities(city_list):
    #delayed_jobs = []
    #delayed_jobs = [delayed(calculate_building_distances_to_roads)(city) for city in city_list]
    #delayed_jobs.append([delayed(produce_blocks)(city) for city in city_list])
    #delayed_jobs.append([calculate_building_distances_to_roads(city) for city in city_list])
    #delayed_jobs.append([produce_blocks(city,YOUR_NAME,grid_size) for city in city_list])
    #results = compute(*delayed_jobs)
    #return results

#preprocess_all_cities(["Nairobi"]) 
#["Abidjan", "Accra", "Bamako", "Belo_Horizonte", "Bogota", "Campinas", "Cape_Town", "Lagos","Luanda", "Nairobi","Medellin"]
#["Abidjan", "Accra", "Nairobi", "Bamako", "Belo_Horizonte", "Bogota", "Campinas", "Cape_Town", "Lagos","Luanda", "Nairobi", "Medellin"])

end_time = time.time()  # End the timer
elapsed_time = end_time - start_time

print(f"Tasks completed in {elapsed_time:.2f} seconds.")

In [ ]:
# PRODUCE AZIMUTHS

start_time = time.time()  # Start the timer


city_list = cities = ["Abidjan", "Accra", "Bamako", "Belo_Horizonte", "Bogota", "Campinas", "Cape_Town", "Lagos","Luanda", "Nairobi", "Medellin"]
#["Abidjan", "Accra", "Nairobi", "Bamako", "Belo_Horizonte", "Bogota", "Campinas", "Cape_Town", "Lagos","Luanda", "Nairobi", "Medellin"])

delayed_jobs = []
delayed_jobs.append([produce_azimuths(city, YOUR_NAME, 200) for city in city_list])
results = compute(*delayed_jobs)

end_time = time.time()  # End the timer
elapsed_time = end_time - start_time

print(f"Tasks completed in {elapsed_time:.2f} seconds.")

In [6]:
def read_validation_set_s3(bucket_prefix = "wri-cities-sandbox/identifyingLandSubdivisions/data/validation/Land_Use/102_cities",
                           base = "Final_102Cities"):
    
    # or use 9_cities/Sub_Saharan_9_cities_and_LAC_3_cities_Land_use
    
    fs = s3fs.S3FileSystem(anon=False)

    # All components of the shapefile
    suffixes = [".shp", ".dbf", ".shx", ".prj", ".cpg"]

    local_dir = "/tmp/land_use"
    os.makedirs(local_dir, exist_ok=True)

    for suf in suffixes:
        s3_path = f"{bucket_prefix}/{base}{suf}"
        local_path = os.path.join(local_dir, base + suf)
        with fs.open(s3_path, "rb") as fsrc, open(local_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)

    # Now read from local path
    local_shp = os.path.join(local_dir, base + ".shp")
    validation_set = gpd.read_file(local_shp)

    return validation_set

validation_set = read_validation_set_s3()

mix = "Democratic_Republic_of_the_Congo_Republic_of_Congo"

# Brazzaville -> Kinshasa
validation_set["City"] = validation_set["City"].mask(validation_set["City"].eq("Brazzaville"), "Kinshasa")

# Set Country to the mixed one for (original) Kinshasa and Brazzaville rows
validation_set["Country"] = validation_set["Country"].mask(
    validation_set["City"].eq("Kinshasa"),  # after the line above, Brazzaville is now Kinshasa too
    mix
)

validation_set.loc[validation_set["City"] == "N_Djamena", "Country"] = "Chad"

cities = validation_set.City.unique()

validation_set[validation_set.Country.isna()]


MAIN_PATH = "s3://wri-cities-sandbox/identifyingLandSubdivisions/data"
OUTPUT_PATH = f"{MAIN_PATH}/output"
OUTPUT_PATH_RASTER = f"{OUTPUT_PATH}/raster"


repl = {
        "United Republic of Tanzania": "Tanzania",
        "Congo": "Republic of Congo",
        "Pointe-Noire": "Pointe Noire",
        "Côte d'Ivoire": "Cote d Ivoire",
        "CÃ´te d'Ivoire": "Cote d Ivoire",
        "Cã´Te-D'Ivoire" : "Cote d Ivoire",
        "Mbour": "MBour",
        "Manzin": "Manzini",
        "Burkina-Faso":"Burkina Faso",
        "Sierra-Leone":"Sierra Leone",
        "South-Africa": "South Africa",
        "Guinea-Bissau": "Guinea Bissau",
        "Ido ekiti":"Ido Ekiti",
        'Port-gentil':"Port Gentil"
            }

validation_set[["City", "Country"]] = validation_set[["City", "Country"]].replace(repl)


cities = validation_set[['City','Country']].drop_duplicates().apply(lambda x: "_".join(x['City'].split())+'__'+"_".join(x['Country'].split()), axis=1).values 

cities = list(set(cities) - set(['Banki__Nigeria', 'Maseru__South_Africa', 'Lome__Ghana','N_Djamena__Cameroon']))


In [ ]:
len(cities)

In [7]:
start_time = time.time()  # Start the timer

#cities = ["Nairobi"]#"Accra", "Bamako", "Belo_Horizonte", "Bogota","Campinas", "Cape_Town","Abidjan","Luanda",  ]
#cities = ['Medellin']
#"Lagos"
cities = [city.replace(' ', '_') for city in cities]

grid_size=100

tasks = []
for city in cities:
    tasks.append(building_and_intersection_metrics(city,YOUR_NAME))
    tasks.append(building_distance_metrics(city, YOUR_NAME))
    tasks.append(compute_m6_m7(city, YOUR_NAME))
    tasks.append(metrics_roads_intersections(city, YOUR_NAME))
    tasks.append(metrics_k_blocks(city, YOUR_NAME))

results = compute(*tasks)

end_time = time.time()  # End the timer
elapsed_time = end_time - start_time

print(f"Tasks completed in {elapsed_time:.2f} seconds.")

Mtwara__Tanzania EPSG: 32737
Masvingo__Zimbabwe EPSG: 32736
Francistown__Botswana EPSG: 32735
Boditi__Ethiopia EPSG: 32637
Bouake__Cote_d_Ivoire EPSG: 32630
Moundou__Chad EPSG: 32633
Embu__Kenya EPSG: 32737
Faya_Largeau__Chad EPSG: 32634
Nouakchott__Mauritania EPSG: 32628
Bujumbura__Burundi EPSG: 32735
Porto_Novo__Benin EPSG: 32631
Monrovia__Liberia EPSG: 32629Yaounde__Cameroon EPSG: 32632

Maputo__Mozambique EPSG: 32736


/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------------+-------------+
| Merge columns  | left dtype | right dtype |
+----------------+------------+-------------+
| ('u', 'osmid') | float64    | int64       |
+----------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------------+-------------+
| Merge columns  | left dtype | right dtype |
+----------------+------------+-------------+
| ('v', 'osmid') | float64    | int64       |
+----------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/data

Kissidougou__Guinea EPSG: 32629
Pointe_Noire__Republic_of_Congo EPSG: 32732
Abidjan__Cote_d_Ivoire EPSG: 32630
Bata__Equatorial_Guinea EPSG: 32632
Serrekunda__Gambia EPSG: 32628
Moroni__Comoros EPSG: 32738
Bondoukou__Cote_d_Ivoire EPSG: 32630
Bunia__Democratic_Republic_of_the_Congo EPSG: 32636
Kakanda__Democratic_Republic_of_the_Congo EPSG: 32735


/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 281232 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------------+-------------+
| Merge columns  | left dtype | right dtype |
+----------------+------------+-------------+
| ('u', 'osmid') | float64    | int64       |
+----------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------

Manzini__Swaziland EPSG: 32736


...

Accra__Ghana EPSG: 32630
Koidu_Town__Sierra_Leone EPSG: 32629


/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------------+-------------+
| Merge columns  | left dtype | right dtype |
+----------------+------------+-------------+
| ('u', 'osmid') | float64    | int64       |
+----------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/multi.py:169: UserWarning: Merging dataframes with merge column data type mismatches: 
+----------------+------------+-------------+
| Merge columns  | left dtype | right dtype |
+----------------+------------+-------------+
| ('v', 'osmid') | float64    | int64       |
+----------------+------------+-------------+
Cast dtypes explicitly to avoid unexpected results.
  warnings.warn(


Tenkodogo__Burkina_Faso EPSG: 32630


/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 135282 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return g.apply(func, *args, **kwargs)
/opt/anaconda3/envs/subdivisions2/lib/python3.12/site-packages/dask/dataframe/groupby.py:116: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is de

Magaria__Niger EPSG: 32632
Pointe_Noire__Republic_of_Congo EPSG: 32732
Gaborone__Botswana EPSG: 32735
Harare__Zimbabwe EPSG: 32736
Windhoek__Namibia EPSG: 32733
Asmara__Eritrea EPSG: 32637
Cape_Town__South_Africa EPSG: 32734
Lilongwe__Malawi EPSG: 32736
Kampala__Uganda EPSG: 32636
Monrovia__Liberia EPSG: 32629
Nzerekore__Guinea EPSG: 32629
Embu__Kenya EPSG: 32737
Buurhakaba__Somalia EPSG: 32638
Sikasso__Mali EPSG: 32630
Luanda__Angola EPSG: 32733
Campinas__Brazil EPSG: 32723
Jimma__Ethiopia EPSG: 32637
Mbeya__Tanzania EPSG: 32736
Banki__Cameroon EPSG: 32633
Nouakchott__Mauritania EPSG: 32628
Bogota__Colombia EPSG: 32618
Keren__Eritrea EPSG: 32637
Bondoukou__Cote_d_Ivoire EPSG: 32630
Edendale__South_Africa EPSG: 32736
Maputo__Mozambique EPSG: 32736
Mogadishu__Somalia EPSG: 32638
Bulawayo__Zimbabwe EPSG: 32735
Johannesburg__South_Africa EPSG: 32735
MBour__Senegal EPSG: 32628
Conakry__Guinea EPSG: 32628
Porto_Novo__Benin EPSG: 32631
Bunia__Democratic_Republic_of_the_Congo EPSG: 32636
Fomb

In [ ]:
# HERE I NEED TO COMBINE METRIC FILES INTO A SINGLE ONE PER CITY